# Lab 04 — Python patterns for AI engineering

Original OfferReady lab. The Python that actually shows up in production GenAI
code: validating model I/O, resilient external calls, and concurrent fan-out.
Runs with only the standard library (plus optional pydantic).

## 1. Validate model output at the boundary

Turn "hope the JSON is right" into a guarantee. Uses pydantic if present, else a
plain fallback so the lab always runs.

In [ ]:
import json

try:
    from pydantic import BaseModel, Field, ValidationError

    class Analysis(BaseModel):
        summary: str
        risk_score: int = Field(ge=0, le=100)
        tags: list[str] = []

    def parse(raw):
        try:
            return Analysis.model_validate_json(raw)
        except ValidationError as e:
            print("schema errors:", e.error_count())
            return None
except ImportError:
    def parse(raw):
        d = json.loads(raw)
        assert 0 <= d["risk_score"] <= 100
        return d

print(parse('{"summary": "ok", "risk_score": 20, "tags": ["a"]}'))
print(parse('{"summary": "bad", "risk_score": 999}'))  # rejected

## 2. Resilient calls: timeout + backoff

Every model/API call fails sometimes. Retry transient errors with exponential
backoff + jitter; don't retry errors you caused.

In [ ]:
import time, random

class Transient(Exception):
    pass

def call_with_retry(fn, attempts=4, base=0.2):
    for i in range(attempts):
        try:
            return fn()
        except Transient:
            if i == attempts - 1:
                raise
            time.sleep(base * (2 ** i) + random.uniform(0, 0.05))

calls = {"n": 0}
def flaky():
    calls["n"] += 1
    if calls["n"] < 3:
        raise Transient("429")
    return "ok after retries"

print(call_with_retry(flaky), "in", calls["n"], "tries")

## 3. Concurrent fan-out for I/O-bound work

Many model/API calls at once = async. This is I/O concurrency, not CPU
parallelism (for CPU work, use processes).

In [ ]:
import asyncio

async def fake_call(text):
    await asyncio.sleep(0.05)          # simulate network latency
    return text.upper()

async def main():
    items = ["a", "b", "c", "d"]
    return await asyncio.gather(*(fake_call(t) for t in items))

# In a notebook the event loop is already running, so await directly.
await main()

## Takeaways

- Validate LLM/API data at the boundary (pydantic) — never trust raw JSON.
- Timeout + backoff every external call; make retries idempotent.
- Async for I/O-bound fan-out; processes for CPU-bound work.

See the Study Guide, Chapter 3 (Python for AI engineering).